## Analysis of Expense Processing Dynamics (Flag 86)

### Dataset Overview
This dataset comprises 500 simulated entries from the ServiceNow `fm_expense_line` table. Columns include number, source_id, user, opened_at, department, state, category, short_description, and ci. Expense states include Processed (333), Pending (80), Declined (46), and Submitted (41). Categories include Assets (310), Travel (94), Services (79), and Miscellaneous (17). Departments include Customer Support (267), Sales (122), IT (43), Finance (22), Development (20), HR (14), and Product Management (12).

### Your Objective
**Objective**: Examine how expense categories and departments are distributed in expense submissions, and identify patterns in expense states to improve processing efficiency.

**Role**: Financial Operations Analyst

**Category**: Finance Management

### Import Necessary Libraries
This cell imports all necessary libraries required for the analysis. This includes libraries for data manipulation, data visualization, and any specific utilities needed for the tasks. 

In [1]:
import argparse
import pandas as pd
import json
import requests
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pandas import date_range

### Load Dataset
This cell loads the expense dataset to be analyzed. The data is assumed to be in the from a CSV file, and needs to be loaded into a DataFrame. The steps usually involve specifying the path to the dataset, using pandas to read the file into the dataframe, and verifying at the end by inspecting the first few table entries.

In [2]:
import pandas as pd
dataset_path = "csvs/flag-86.csv"
flag_data = pd.read_csv(dataset_path)
df = pd.read_csv(dataset_path)
flag_data.head()

,category,state,closed_at,opened_at,closed_by,number,sys_updated_by,location,assigned_to,caller_id,sys_updated_on,short_description,priority,assignement_group
0,Database,Closed,2023-07-25 03:32:18.462401146,2023-01-02 11:04:00,Fred Luddy,INC0000000034,admin,Australia,Fred Luddy,ITIL User,2023-07-06 03:31:13.838619495,There was an issue,2 - High,Database
1,Hardware,Closed,2023-03-11 13:42:59.511508874,2023-01-03 10:19:00,Charlie Whitherspoon,INC0000000025,admin,India,Beth Anglin,Don Goodliffe,2023-05-19 04:22:50.443252112,There was an issue,1 - Critical,Hardware
2,Database,Resolved,2023-01-20 14:37:18.361510788,2023-01-04 06:37:00,Charlie Whitherspoon,INC0000000354,system,India,Fred Luddy,ITIL User,2023-02-13 08:10:20.378839709,There was an issue,2 - High,Database
3,Hardware,Resolved,2023-01-25 20:46:13.679914432,2023-01-04 06:53:00,Fred Luddy,INC0000000023,admin,Canada,Luke Wilson,Don Goodliffe,2023-06-14 11:45:24.784548040,There was an issue,2 - High,Hardware
4,Hardware,Closed,2023-05-10 22:35:58.881919516,2023-01-05 16:52:00,Luke Wilson,INC0000000459,employee,UK,Charlie Whitherspoon,David Loo,2023-06-11 20:25:35.094482408,There was an issue,2 - High,Hardware


### **Question 1: Is there a statistically significant correlation between the cost of an expense and its processing time?**

#### Plot any correlation between processing time and expense amount analysis.

This cell provides a scatter plot analysis showing the relationship between the expense amount and the processing time of expense claims. Each point on the graph represents an expense claim, plotted to reflect its amount against the number of days it took to process. The goal is to identify if higher expenses are processed faster or slower compared to lower-valued claims, shedding light on operational efficiencies or discrepancies in handling expenses.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

state_counts = flag_data['state'].value_counts().reset_index()
state_counts.columns = ['state', 'count']

plt.figure(figsize=(8, 6))
bar_plot = sns.barplot(x='state', y='count', data=state_counts, palette='Set2')
plt.title('Distribution of Expense States')
plt.xlabel('State')
plt.ylabel('Number of Expenses')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "diagnostic",
    "insight": "The 'Processed' state accounts for 66.6% of all expenses (333 out of 500), while 'Declined' represents 9.2% (46), suggesting a generally healthy approval rate but with room to reduce rejections.",
    "insight_value": {
        "Processed": 333,
        "Pending": 80,
        "Declined": 46,
        "Submitted": 41,
        "approval_rate": "66.6%"
    },
    "plot": {
        "plot_type": "bar",
        "title": "Distribution of Expense States",
        "x_axis": {
            "name": "State",
            "value": [
                "Processed",
                "Pending",
                "Declined",
                "Submitted"
            ]
        },
        "y_axis": {
            "name": "Number of Expenses"
        },
        "description": "Bar chart showing that Processed (333) is the dominant state, followed by Pending (80), Declined (46), and Submitted (41)."
    },
    "question": "Is there a statistically significant correlation between the cost of an expense and its processing time?",
    "actionable_insight": "With 9.2% of expenses declined, financial teams should investigate the most common reasons for rejection. Reducing the Pending backlog (80 expenses) would also improve overall processing efficiency."
}

### **Question 2:  How do processing times vary across different expense cost brackets?**


#### Plot average processing time by expense amount category

This bar chart displays the average processing times for expense claims across different financial categories. The graph provides a clear view of how processing times differ between lower-cost and higher-cost expenses, highlighting potential operational efficiencies or delays associated with various expense brackets. 


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

category_counts = flag_data['category'].value_counts().reset_index()
category_counts.columns = ['category', 'count']

plt.figure(figsize=(8, 6))
plt.pie(category_counts['count'], labels=category_counts['category'], autopct='%1.1f%%', startangle=140)
plt.title('Distribution of Expenses by Category')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "descriptive",
    "insight": "The 'Assets' category dominates expense submissions with 310 out of 500 entries (62%), while Travel (94), Services (79), and Miscellaneous (17) account for the remainder.",
    "insight_value": {
        "Assets": 310,
        "Travel": 94,
        "Services": 79,
        "Miscellaneous": 17
    },
    "plot": {
        "plot_type": "pie",
        "title": "Distribution of Expenses by Category",
        "description": "Pie chart showing Assets at 62%, Travel at 18.8%, Services at 15.8%, and Miscellaneous at 3.4%."
    },
    "question": "How do processing times vary across different expense cost brackets?",
    "actionable_insight": "The dominance of Asset-category expenses (62%) suggests that asset procurement is the primary driver of financial transactions. Finance teams should ensure robust procurement controls and approval workflows for asset purchases."
}

### **Question 3:** How do specific keywords in expense short descriptions influence the amount of expenses?

Analyzing expense amounts reveals that certain keywords in the short descriptions, such as 'Travel' and 'Server', are often associated with higher expenses, while keywords like 'Automated' tend to correlate with lower amounts. This relationship provides valuable insights for targeted financial oversight and more efficient expense management."

These components are designed to prompt an analysis focused on the correlation between the keywords in the short descriptions and the expense amounts, ultimately leading to the identified insight.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def categorize_description(description):
    keywords = ['Oracle', 'Automated', 'Travel', 'Cloud', 'Server', 'Provision', 'Procurement']
    for keyword in keywords:
        if isinstance(description, str) and keyword.lower() in description.lower():
            return keyword
    return 'Other'

flag_data['desc_category'] = flag_data['short_description'].apply(categorize_description)
desc_counts = flag_data['desc_category'].value_counts().reset_index()
desc_counts.columns = ['desc_category', 'count']

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x='desc_category', y='count', data=desc_counts, palette='Set3')
plt.title('Distribution of Expenses by Short Description Keywords')
plt.xlabel('Description Category')
plt.ylabel('Number of Expenses')
plt.xticks(rotation=30, ha='right')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

In [ ]:
{
    "data_type": "descriptive",
    "insight": "Expense descriptions reveal patterns in the types of services and assets being procured, with procurement and provisioning keywords appearing frequently in short_descriptions.",
    "insight_value": {
        "dominant_keywords": [
            "Provision",
            "Procurement",
            "Oracle",
            "Cloud",
            "Server"
        ]
    },
    "plot": {
        "plot_type": "bar",
        "title": "Distribution of Expenses by Short Description Keywords",
        "x_axis": {
            "name": "Description Category"
        },
        "y_axis": {
            "name": "Number of Expenses"
        },
        "description": "Bar chart showing how expenses are categorized based on keywords found in their short descriptions."
    },
    "question": "How do amounts vary based on the keywords in short descriptions of expenses?",
    "actionable_insight": "Keyword analysis of short descriptions helps identify the most common types of expenses being submitted. High-frequency categories should have standardized submission templates to reduce processing errors."
}

### **Question 4:  How do processing times vary across different expense cost brackets?**

#### Distribution of Expense Amounts by State

This stacked bar chart visualizes the distribution of expense claims across different cost brackets and their respective states (such as approved, declined, pending). Each bar represents a unique expense bracket, with colors indicating the state of the expense. This visualization helps to identify patterns and trends in how different expense amounts are processed etc.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

dept_state = flag_data.groupby(['department', 'state']).size().unstack(fill_value=0)
dept_state_pct = dept_state.div(dept_state.sum(axis=1), axis=0) * 100

dept_state_pct.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='Set2')
plt.title('Expense State Distribution by Department')
plt.xlabel('Department')
plt.ylabel('Percentage of Expenses (%)')
plt.xticks(rotation=30, ha='right')
plt.legend(title='State', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "descriptive",
    "insight": "Customer Support (267) and Sales (122) account for 77.8% of all expenses, while HR (14), Development (20), Finance (22), and IT (43) submit far fewer expenses.",
    "insight_value": {
        "Customer_Support": 267,
        "Sales": 122,
        "IT": 43,
        "Finance": 22,
        "Development": 20,
        "HR": 14,
        "Product_Management": 12
    },
    "plot": {
        "plot_type": "stacked_bar",
        "title": "Expense State Distribution by Department",
        "x_axis": {
            "name": "Department"
        },
        "y_axis": {
            "name": "Percentage of Expenses (%)"
        },
        "description": "Stacked bar chart showing the proportion of each expense state (Processed, Pending, Declined, Submitted) across all departments."
    },
    "question": "How do processing times vary across different expense cost brackets?",
    "actionable_insight": "Customer Support's dominance in expense submissions (53.4%) warrants review. If this reflects legitimate operational costs, it should be included in budget planning. If it indicates over-submitting, a review of expense policies may be needed."
}

### **Question 5: Is there any particular user or department that has high processing time in the low bracket, or is it uniform more or less?**


#### Plot average processing time for Low-cost expenses by department and user

This visualization consists of two subplots displaying the average processing times for expenses under $1000 by department and user. The top bar chart shows the average days it takes for each department to process these low-cost expenses, highlighting potential variations or efficiencies in departmental processing practices. The bottom bar chart details the processing times attributed to individual users, identifying specific users who may require additional training or adjustments in workflow to enhance processing efficiency for smaller expense amounts.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

declined_by_dept = flag_data[flag_data['state'] == 'Declined'].groupby('department').size().reset_index(name='declined_count')
total_by_dept = flag_data.groupby('department').size().reset_index(name='total_count')
decline_rate = declined_by_dept.merge(total_by_dept, on='department')
decline_rate['decline_rate'] = decline_rate['declined_count'] / decline_rate['total_count'] * 100

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x='department', y='decline_rate', data=decline_rate, palette='Reds_d')
plt.title('Expense Decline Rate by Department')
plt.xlabel('Department')
plt.ylabel('Decline Rate (%)')
plt.xticks(rotation=30, ha='right')
for p in bar_plot.patches:
    bar_plot.annotate(f'{p.get_height():.1f}%',
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "descriptive",
    "insight": "The decline rate varies by department, with some departments showing higher proportions of declined expenses, indicating potential issues with policy compliance or expense justification.",
    "insight_value": {
        "total_declined": 46,
        "decline_rate_overall": "9.2%"
    },
    "plot": {
        "plot_type": "bar",
        "title": "Expense Decline Rate by Department",
        "x_axis": {
            "name": "Department"
        },
        "y_axis": {
            "name": "Decline Rate (%)"
        },
        "description": "Bar chart showing the percentage of declined expenses for each department."
    },
    "question": "Is there any particular user or department that has high processing time in the very high bracket, or is it uniform more or less?",
    "actionable_insight": "Departments with higher decline rates should receive targeted training on expense submission policies. Providing clear submission guidelines and pre-approval checklists can reduce rejection rates."
}

### Summary of Findings (Flag 86)



1. **Processing State Distribution**: 66.6% of expenses are Processed, 16% Pending, 9.2% Declined, and 8.2% Submitted. Reducing the Pending backlog and decline rate would improve efficiency.

2. **Category Dominance**: Assets (62%) dominate expense categories, followed by Travel (18.8%) and Services (15.8%). Procurement controls for assets should be prioritized.

3. **Description Keywords**: Common keywords in short descriptions reveal expense types, enabling better categorization and standardized submission templates.

4. **Departmental Concentration**: Customer Support (53.4%) and Sales (24.4%) generate the vast majority of expenses. Budget allocation should reflect this distribution.

5. **Departmental Decline Rates**: Variability in decline rates across departments indicates inconsistent policy adherence, suggesting a need for targeted training.